# Testing for B0s signal drift in Obelix and Asterix
----------------------------------------------------------
I wanna plot signal drift in Obelix and Asterix, as continuos line to show how it change across time and spatially, i will do it as Anterior/Middle and front masks 
1) Get the masks for each participant
2) plot Asterix/Obelix drift across time and space 

In [9]:
import os 
import numpy as np 
import subprocess as sub 
import nibabel as nib
import matplotlib.pyplot as plt
import nilearn as nil 
from nilearn.image.image import mean_img #Estimate mean of a 4D volume --> mean_haxby = mean_img(func_filename)
from nilearn.plotting import plot_roi, show
from nilearn.maskers import NiftiMasker #Create a mask

# ===========================================================================
# CONFIGURATION – tweak these defaults; the interactive loop will let you
# refine them subject-by-subject
# =========================

home=r"/home/malberti/wks14/temp/FF_DWI_Drift"
derivatives=r"/home/malberti/wks14/temp/FF_DWI_Drift/derivatives"      

In [ ]:
PANORAMIX = f"/home/malberti/Unix_Folders/Asterix_Obelix_Population_Template/Panoramix_Template_Tmean.nii"
metric    = "MD"

# Default cube size (px, py, pz)
DEFAULT_CUBE_SIZE = (15, 15, 15)

# Default per-ROI offsets from the auto-computed centres.
# Adjust sign/magnitude here, or interactively when prompted.
DEFAULT_OFFSETS = {
    "ant":  {"dx": 0, "dy": 23,  "dz":  5},   # anterior  (BrBG)
    "mid":  {"dx": 0, "dy": 15,  "dz": -15},  # middle    (RdGy)
    "post": {"dx": 0, "dy": 13,  "dz":  5},   # posterior (Blues_r)
}

# ── Input ────────────────────────────────────────────────────────────────────
Asterix_c = [16, 17, 21, 25, 26, 29, 30, 32, 33, 34, 35, 36, 39, 40]   # ses-01
Asterix_w = [18, 19, 22, 23, 24, 27, 28, 31, 37, 38, 9999, 9998]       # ses-13

conditions = {
    "Asterix_c": (Asterix_c, "01"),
    "Asterix_w": (Asterix_w, "13"),
}


# ===========================================================================
# HELPERS
# ===========================================================================

def build_masks(mask, offsets, cube_size):
    """Return (ant, mid, post) binary mask arrays given current offsets."""
    X, Y, Z    = mask.shape[:3]
    px, py, pz = cube_size

    x_center = X // 2 - px // 2
    z_center = Z // 2 - pz // 2

    starts = {
        "ant":  (x_center + offsets["ant"]["dx"],
                 Y // 2   + offsets["ant"]["dy"] - py // 2,
                 z_center + offsets["ant"]["dz"]),

        "mid":  (x_center + offsets["mid"]["dx"],
                 Y // 3   + offsets["mid"]["dy"] - py // 2,
                 z_center + offsets["mid"]["dz"]),

        "post": (x_center + offsets["post"]["dx"],
                 Y // 6   + offsets["post"]["dy"] - py // 2,
                 z_center + offsets["post"]["dz"]),
    }

    masks = {}
    for key, (sx, sy, sz) in starts.items():
        m = np.zeros((X, Y, Z), dtype=np.uint8)
        # clamp to valid range
        sx = max(0, min(sx, X - px))
        sy = max(0, min(sy, Y - py))
        sz = max(0, min(sz, Z - pz))
        m[sx:sx+px, sy:sy+py, sz:sz+pz] = 1
        masks[key] = m * mask

    return masks, starts


def save_tmp_masks(masks, affine, tmp_dir):
    """Save masks to a temp location; return dict of paths."""
    paths = {}
    for key, arr in masks.items():
        img  = nib.Nifti1Image(arr, affine)
        path = os.path.join(tmp_dir, f"_tmp_cube_{key}.nii.gz")
        nib.save(img, path)
        paths[key] = path
    return paths


def open_fsleyes(ref_path, mask_paths, template_path=PANORAMIX):
    """Launch FSLeyes with template + ref + masks and block until closed."""
    cmd = [
        "fsleyes",
        template_path,                              # background template
        ref_path,      "-cm", "greyscale",          # subject MD map
        mask_paths["ant"],  "-cm", "copper",        # anterior  → warm
        mask_paths["mid"],  "-cm", "red-yellow",    # middle    → red
        mask_paths["post"], "-cm", "cool",          # posterior → blue
    ]
    print("\n  Opening FSLeyes … close the window when you are done reviewing.\n")
    subprocess.run(cmd)   # blocks until FSLeyes is closed


def ask_continue():
    """Ask the user whether masks are OK. Returns True if approved."""
    print("\n  Are the masks OK?")
    print("    [y] Yes – save and continue to next subject")
    print("    [n] No  – stop here so you can adjust the coordinates in the script")
    answer = input("  → ").strip().lower()
    return answer == "y"


# ===========================================================================
# MAIN LOOP
# ===========================================================================

for condition, (sub_ids, ses) in conditions.items():
    for sid in sub_ids:

        pid    = f"sub-{sid:02d}"
        subjid = f"{pid}_ses-{ses}"
        base      = os.path.join(home, "derivatives", pid, f"ses-{ses}", "dwi")
        coreg_dir = os.path.join(base, "index2TEMPLATE")
        tmp_dir   = os.path.join(coreg_dir, "_tmp_masks")
        os.makedirs(tmp_dir, exist_ok=True)

        # --- load data ---
        mask_path = r"/home/malberti/Unix_Folders/Asterix_Obelix_Population_Template/Panoramix_Template_Tmean_mask.nii" #os.path.join(coreg_dir, f"{subjid}_prep_{metric}2PANORAMIX_mask.nii")
        ref_path  = os.path.join(coreg_dir, f"{subjid}_prep_{metric}2PANORAMIX.nii")

        if not os.path.exists(ref_path):
            print(f"✗  MISSING data for {subjid}, skipping")
            print(ref_path)
            continue

        mask_IMG = nib.load(mask_path)
        ref_IMG  = nib.load(ref_path)

        mask   = mask_IMG.get_fdata()
        ref    = mean_img(ref_IMG)
        affine = mask_IMG.affine

        # save the mean ref to disk so FSLeyes can load it
        ref_tmp_path = os.path.join(tmp_dir, f"_tmp_ref_{subjid}.nii.gz")
        nib.save(ref, ref_tmp_path)

        # --- start with default offsets ---
        offsets    = {k: dict(v) for k, v in DEFAULT_OFFSETS.items()}
        cube_size  = DEFAULT_CUBE_SIZE
        approved   = False

        print(f"\n{'='*60}")
        print(f"  Subject: {subjid}  ({condition})")
        print(f"{'='*60}")

        while not approved:
            # build & save temp masks
            masks, starts = build_masks(mask, offsets, cube_size)
            mask_paths    = save_tmp_masks(masks, affine, tmp_dir)

            print(f"\n  Current ROI starts (voxel coords):")
            for roi, s in starts.items():
                print(f"    {roi:4s}  x={s[0]:4d}  y={s[1]:4d}  z={s[2]:4d}")

            # open FSLeyes for visual check
           # open_fsleyes(ref_tmp_path, mask_paths)
            # ask the user
            approved = True

        # ---------------------------------------------------------------
        # User approved – save final masks
        # ---------------------------------------------------------------
        masks_output = os.path.join(derivatives, pid, f"ses-{ses}", "dwi", "masks")
        os.makedirs(masks_output, exist_ok=True)

        label_map = {"ant": "anterior", "mid": "mid", "post": "posterior"}
        for key, arr in masks.items():
            img   = nib.Nifti1Image(arr, affine)
            fname = f"{subjid}_cube_mask-{label_map[key]}.nii.gz"
            nib.save(img, os.path.join(masks_output, fname))
            print(f"  Saved → {fname}")

        # clean up temp files
        for p in mask_paths.values():
            os.remove(p)
        os.remove(ref_tmp_path)

        print(f"\n  ✓ {subjid} done.\n")

## Create MASK in subject space
-------------------------------
The code below create the cubic masks on subject space and plot them 

In [10]:
import os
import subprocess
import numpy as np
import nibabel as nib
from nilearn.image import mean_img

# ===========================================================================
# CONFIGURATION – tweak these defaults; the interactive loop will let you
# refine them subject-by-subject
# ===========================================================================

PANORAMIX = f"/home/malberti/Unix_Folders/Asterix_Obelix_Population_Template/Panoramix_Template_Tmean.nii"
metric    = "MD"

# Default cube size (px, py, pz)
DEFAULT_CUBE_SIZE = (15, 15, 15)

# Default per-ROI offsets from the auto-computed centres.
# Adjust sign/magnitude here, or interactively when prompted.
DEFAULT_OFFSETS = {
    "ant":  {"dx": 0, "dy": 20,  "dz":  5},   # anterior  (BrBG)
    "mid":  {"dx": 0, "dy": 13,  "dz": -15},  # middle    (RdGy)
    "post": {"dx": 0, "dy": 11,  "dz":  5},   # posterior (Blues_r)
}

# ── Input ────────────────────────────────────────────────────────────────────
Asterix_c = [16, 17, 21, 25, 26, 29, 30, 32, 33, 34, 35, 36, 39, 40]   # ses-01
Asterix_w = [18, 19, 22, 23, 24, 27, 28, 31, 37, 38, 9999, 9998]       # ses-13

conditions = {
    "Asterix_c": (Asterix_c, "01"),
    "Asterix_w": (Asterix_w, "13"),
}


# ===========================================================================
# HELPERS
# ===========================================================================

def build_masks(mask, offsets, cube_size):
    """Return (ant, mid, post) binary mask arrays given current offsets."""
    X, Y, Z    = mask.shape[:3]
    px, py, pz = cube_size

    x_center = X // 2 - px // 2
    z_center = Z // 2 - pz // 2

    starts = {
        "ant":  (x_center + offsets["ant"]["dx"],
                 Y // 2   + offsets["ant"]["dy"] - py // 2,
                 z_center + offsets["ant"]["dz"]),

        "mid":  (x_center + offsets["mid"]["dx"],
                 Y // 3   + offsets["mid"]["dy"] - py // 2,
                 z_center + offsets["mid"]["dz"]),

        "post": (x_center + offsets["post"]["dx"],
                 Y // 6   + offsets["post"]["dy"] - py // 2,
                 z_center + offsets["post"]["dz"]),
    }

    masks = {}
    for key, (sx, sy, sz) in starts.items():
        m = np.zeros((X, Y, Z), dtype=np.uint8)
        # clamp to valid range
        sx = max(0, min(sx, X - px))
        sy = max(0, min(sy, Y - py))
        sz = max(0, min(sz, Z - pz))
        m[sx:sx+px, sy:sy+py, sz:sz+pz] = 1
        masks[key] = m * mask

    return masks, starts


def save_tmp_masks(masks, affine, tmp_dir):
    """Save masks to a temp location; return dict of paths."""
    paths = {}
    for key, arr in masks.items():
        img  = nib.Nifti1Image(arr, affine)
        path = os.path.join(tmp_dir, f"_tmp_cube_{key}.nii.gz")
        nib.save(img, path)
        paths[key] = path
    return paths


def open_fsleyes(ref_path, mask_paths, template_path=PANORAMIX):
    """Launch FSLeyes with template + ref + masks and block until closed."""
    cmd = [
        "fsleyes",
        #template_path,                              # background template
        ref_path,      "-cm", "greyscale",          # subject MD map
        mask_paths["ant"],  "-cm", "copper",        # anterior  → warm
        mask_paths["mid"],  "-cm", "red-yellow",    # middle    → red
        mask_paths["post"], "-cm", "cool",          # posterior → blue
    ]
    print("\n  Opening FSLeyes … close the window when you are done reviewing.\n")
    subprocess.run(cmd)   # blocks until FSLeyes is closed


def ask_continue():
    """Ask the user whether masks are OK. Returns True if approved."""
    print("\n  Are the masks OK?")
    print("    [y] Yes – save and continue to next subject")
    print("    [n] No  – stop here so you can adjust the coordinates in the script")
    answer = input("  → ").strip().lower()
    return answer == "y"


# ===========================================================================
# MAIN LOOP
# ===========================================================================

for condition, (sub_ids, ses) in conditions.items():
    for sid in sub_ids:

        pid    = f"sub-{sid:02d}"
        subjid = f"{pid}_ses-{ses}"
        base     = os.path.join(home, "derivatives", pid, f"ses-{ses}", "dwi")
        eddy     = os.path.join(base, "eddy")
        mask_dir = os.path.join(base, "mask_dMRI-space")
        tmp_dir  = os.path.join(mask_dir, "_tmp_masks")
        os.makedirs(mask_dir, exist_ok=True)
        os.makedirs(tmp_dir, exist_ok=True)

        # --- load data ---
        mask_path = os.path.join(eddy, f"{subjid}_unwarped_mask.nii")
        ref_path  = os.path.join(eddy, f"{subjid}_unwarped.nii")

        if not (os.path.exists(mask_path) and os.path.exists(ref_path)):
            print(f"✗  MISSING data for {subjid}, skipping")
            continue

        mask_IMG = nib.load(mask_path)
        ref_IMG  = nib.load(ref_path)

        mask   = mask_IMG.get_fdata()
        ref    = mean_img(ref_IMG)
        affine = mask_IMG.affine

        # save the mean ref to disk so FSLeyes can load it
        ref_tmp_path = os.path.join(tmp_dir, f"_tmp_ref_{subjid}.nii.gz")
        nib.save(ref, ref_tmp_path)

        # --- start with default offsets ---
        offsets    = {k: dict(v) for k, v in DEFAULT_OFFSETS.items()}
        cube_size  = DEFAULT_CUBE_SIZE
        approved   = False

        print(f"\n{'='*60}")
        print(f"  Subject: {subjid}  ({condition})")
        print(f"{'='*60}")

        while not approved:
            # build & save temp masks
            masks, starts = build_masks(mask, offsets, cube_size)
            mask_paths    = save_tmp_masks(masks, affine, tmp_dir)

            print(f"\n  Current ROI starts (voxel coords):")
            for roi, s in starts.items():
                print(f"    {roi:4s}  x={s[0]:4d}  y={s[1]:4d}  z={s[2]:4d}")

            # open FSLeyes for visual check
          #  open_fsleyes(ref_tmp_path, mask_paths)
            # ask the user
            approved = True

        # ---------------------------------------------------------------
        # User approved – save final masks
        # ---------------------------------------------------------------
        masks_output = os.path.join(derivatives, pid, f"ses-{ses}", "dwi", "masks")
        os.makedirs(masks_output, exist_ok=True)

        label_map = {"ant": "anterior", "mid": "mid", "post": "posterior"}
        for key, arr in masks.items():
            img   = nib.Nifti1Image(arr, affine)
            fname = f"{subjid}_cube_mask_dMRI-space-{label_map[key]}.nii.gz"
            nib.save(img, os.path.join(masks_output, fname))
            print(f"  Saved → {fname}")

        # clean up temp files
        for p in mask_paths.values():
            os.remove(p)
        os.remove(ref_tmp_path)

        print(f"\n  ✓ {subjid} done.\n")


  Subject: sub-16_ses-01  (Asterix_c)

  Current ROI starts (voxel coords):
    ant   x=  48  y=  68  z=  33
    mid   x=  48  y=  42  z=  13
    post  x=  48  y=  22  z=  33
  Saved → sub-16_ses-01_cube_mask_dMRI-space-anterior.nii.gz
  Saved → sub-16_ses-01_cube_mask_dMRI-space-mid.nii.gz
  Saved → sub-16_ses-01_cube_mask_dMRI-space-posterior.nii.gz

  ✓ sub-16_ses-01 done.


  Subject: sub-17_ses-01  (Asterix_c)

  Current ROI starts (voxel coords):
    ant   x=  48  y=  68  z=  33
    mid   x=  48  y=  42  z=  13
    post  x=  48  y=  22  z=  33
  Saved → sub-17_ses-01_cube_mask_dMRI-space-anterior.nii.gz
  Saved → sub-17_ses-01_cube_mask_dMRI-space-mid.nii.gz
  Saved → sub-17_ses-01_cube_mask_dMRI-space-posterior.nii.gz

  ✓ sub-17_ses-01 done.


  Subject: sub-21_ses-01  (Asterix_c)

  Current ROI starts (voxel coords):
    ant   x=  48  y=  68  z=  33
    mid   x=  48  y=  42  z=  13
    post  x=  48  y=  22  z=  33
  Saved → sub-21_ses-01_cube_mask_dMRI-space-anterior.nii.gz
 